# 06 · The interface

> *"A minimal input–output interface accepts a pasted Bangla passage and
> returns the predicted author alongside the ranked stylistic features that
> most influenced the decision, so predictions remain auditable rather than
> opaque."* — project proposal, §5

Two front ends over the same `Attributor`: this notebook (with a text box if
`ipywidgets` is available) and `app.py` on the command line.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Bengali glyphs need a font that has them; fall back silently if absent.
for _f in ("Nirmala UI", "Vrinda", "Shonar Bangla", "Noto Sans Bengali"):
    try:
        matplotlib.font_manager.findfont(_f, fallback_to_default=False)
        plt.rcParams["font.family"] = _f
        break
    except Exception:
        continue
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight",
                     "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.width", 160, "display.max_columns", 60)

from banglastylo import config
print("project root:", ROOT)

project root: E:\NLP Project\Bangla-author-fingerprinting


In [2]:
from banglastylo.interface import Attributor, render, DEMO_PASSAGE

attributor = Attributor.load()
print("loaded models for:",
      ", ".join(config.AUTHORS[a]["en"]
                for a in attributor.svm.named_steps["clf"].classes_))

loaded models for: Abanindranath Tagore, Bankim Chandra Chattopadhyay, Sarat Chandra Chattopadhyay, Rabindranath Tagore, Ishwar Chandra Vidyasagar


## 6.1 On the built-in demo passage

In [3]:
print(render(attributor.predict(DEMO_PASSAGE), attributor))

PREDICTED AUTHOR : Rabindranath Tagore  (রবীন্দ্রনাথ ঠাকুর)
passage          : 26 tokens, 2 sentences
  ! short passage — stylometric estimates are noisy below ~80 tokens

DISCRIMINATIVE (linear SVM over stylometric features)
  predicted : Rabindranath Tagore
  runner-up : Sarat Chandra Chattopadhyay  (margin 0.386)
  strongest evidence overall (for Rabindranath Tagore over Sarat Chandra Chattopadhyay):
     +0.0940  character n-gram “হা␣”
     +0.0824  character n-gram “াহা␣”
     +0.0810  character n-gram “াহ”
     +0.0742  character n-gram “াইয়”
     +0.0674  character n-gram “াইয”
     +0.0592  character n-gram “িব”
  strongest evidence you can check by eye:
     +0.0513  POS pattern PRON–PART   [value 1.894]
     +0.0418  POS pattern ADP–VERB–NOUN   [value 1.012]
     +0.0374  function word “তাহা”   [value 1.101]
     +0.0314  function word “যাহা”   [value 1.708]
     +0.0297  POS pattern VERB–NOUN–NOUN   [value 0.632]
     +0.0261  POS pattern VERB   [value 1.239]
  evidence poi

## 6.2 Interactive

Type or paste a Bangla passage and press **Attribute**. Longer is better —
see the length curve in notebook 05; below about 80 tokens the estimate is
noisy and the interface says so.

In [4]:
try:
    import ipywidgets as W
    from IPython.display import display, clear_output

    box = W.Textarea(value=DEMO_PASSAGE, placeholder="বাংলা অনুচ্ছেদ…",
                     layout=W.Layout(width="100%", height="150px"))
    button = W.Button(description="Attribute", button_style="success")
    out = W.Output()

    def on_click(_):
        with out:
            clear_output()
            text = box.value.strip()
            if not text:
                print("paste a passage first")
                return
            print(render(attributor.predict(text), attributor))

    button.on_click(on_click)
    display(W.VBox([box, button, out]))
    on_click(None)
except ImportError:
    print("ipywidgets not installed — use the cell below instead")

In [5]:
passage = DEMO_PASSAGE  # <-- paste your own passage here
print(render(attributor.predict(passage), attributor))

PREDICTED AUTHOR : Rabindranath Tagore  (রবীন্দ্রনাথ ঠাকুর)
passage          : 26 tokens, 2 sentences
  ! short passage — stylometric estimates are noisy below ~80 tokens

DISCRIMINATIVE (linear SVM over stylometric features)
  predicted : Rabindranath Tagore
  runner-up : Sarat Chandra Chattopadhyay  (margin 0.386)
  strongest evidence overall (for Rabindranath Tagore over Sarat Chandra Chattopadhyay):
     +0.0940  character n-gram “হা␣”
     +0.0824  character n-gram “াহা␣”
     +0.0810  character n-gram “াহ”
     +0.0742  character n-gram “াইয়”
     +0.0674  character n-gram “াইয”
     +0.0592  character n-gram “িব”
  strongest evidence you can check by eye:
     +0.0513  POS pattern PRON–PART   [value 1.894]
     +0.0418  POS pattern ADP–VERB–NOUN   [value 1.012]
     +0.0374  function word “তাহা”   [value 1.101]
     +0.0314  function word “যাহা”   [value 1.708]
     +0.0297  POS pattern VERB–NOUN–NOUN   [value 0.632]
     +0.0261  POS pattern VERB   [value 1.239]
  evidence poi

## 6.3 The same thing from the command line

```
.venv/Scripts/python.exe app.py --demo
.venv/Scripts/python.exe app.py --text "…বাংলা অনুচ্ছেদ…"
.venv/Scripts/python.exe app.py --file passage.txt --json
```

## 6.4 The structural profile of a passage

Useful when the two paradigms disagree: the raw numbers behind the
attribution, before any model has weighed in.

In [6]:
prof = attributor.profile(passage)
from banglastylo.interface import FEATURE_GLOSS
pd.DataFrame({
    "feature": [FEATURE_GLOSS.get(k, k.replace("_", " ")) for k in prof],
    "value": list(prof.values()),
}).set_index("feature").round(3)

,value
feature,
mean sentence length (tokens),13.000
sentence-length variability,2.000
sentence-length skew,0.000
median sentence length,13.000
sentence-length spread (IQR),2.000
sent len min,11.000
sent len max,15.000
n sentences,2.000
mean word length (characters),4.923
